Итоги 2 пункта, или что мы знаем о данных сейчас:

1) Приращения стационарны. (ADF и KPSS согласованы) => моделируем преращения
2) Распределения имеют тяжёлые хвосты (Избыточный эксцесс, JB отвергает нормальность, хвостовой индекс Хилла = 2-4)
3) По STL наблюдается кластеризация волатильности. (Привет стилизованным фактам из лекции)
4) Корреляции между факторами непостоянны (в кризис растут)

Отсюда следующие **требования к модели**:
1) Моделируем именно приращения.
2) Условное распределение должно иметь тяжёлые хвосты.
3) Необходимо учитывать динамику волатильсности и корреляции между факторами.
4) Модель должна поддерживать оценку методом MLE. 

Приходим к модели GARCH с t-распределением + DCC для динамики корреляции. (в итоге получается модель DCC-GARCH-t).

**Почему DCC-GARCH-t?**

Модель включает:
- Уравнение для условного среднего (обычно константа или малая авторегрессия)
- Уравнение для условной дисперсии (это и есть GARCH) + t - распределение Стьюдента и тяжёлые хвосты для ошибок.
- Уравнение для корреляции (DCC, если модель многомерная)

(!) Без GARCH мы бы предполагали, что волатильность постоянна, что противоречит данным и ведёт к систематической недооценке риска.

В качестве базы возьмём наиболее простой и как правило устойчивый вариант - `GARCH(1, 1)`. Текущая условная дисперсия зависит от одного лага квадрата ошибки и одного лага самой дисперсии.

Переходим к коду и реализации `пункта № 3 и 4`: 



**TO DO: спецификация модели будет позже, всякие формулы там для информативности.**

# Пункт 3.

1) Загружаем данные:

In [41]:
# Импорт библиотек:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from scipy.optimize import minimize
from scipy.stats import t as t_dist
import warnings
warnings.filterwarnings('ignore')

# Загрузка риск-факторов:
import sys
from pathlib import Path
PROJ = Path.cwd()
if not (PROJ / "utils").exists() and (PROJ.parent / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))
from src import config as C

risk_factors = pd.read_parquet(C.DATA_DIR / "risk_factors.parquet")
print(f"Загружено {risk_factors.shape[0]} наблюдений, {risk_factors.shape[1]} факторов")
print(risk_factors.columns.tolist())

Загружено 1235 наблюдений, 8 факторов
['RATE_PC1', 'RATE_PC2', 'RATE_PC3', 'EQ_PC1', 'EQ_PC2', 'EQ_PC3', 'FX_EUR', 'FX_USD']


2) Одномерные `GARCH(1,1)-t` модели:

baseline параметры :
- omega = 0.1*variance (константа в уравнении дисперсии)
- alpha = 0.1 (коэффициент перед ошибкой, показывает, как сильно текущий шок влияет на будущую волатильность.)
- beta = 0.8 (коэффициент перед дисперсией t-1, отражает "память" процесса: чем больше β, тем медленнее затухает волатильность)

Почему такие параметры: 

Это стартовые значения при оптимизации. Впоследующем они меняются.
- alpha и beta: типичные соотношения для финансовых временных рядов. α обычно находится в диапазоне 0.05–0.2, а β обычно в диапазоне 0.7–0.95.
- omega = 0.1*variance - начальное приближение для численной оптимизации. В процессе оптимизации (MLE) это значение будет скорректировано в соответствии с данными.

**Что делаем:** для каждого риск фактора оцениваем свою GARCH(1,1)-t и GARCH(1, 1) с нормальным распределением. Сравним по информационным моделям, какая модель лучше, тем самым зафиксировав и подкрепив свой выбор:

In [46]:
# Словари для хранения результатов
garch_models = {}
garch_params = {}
std_residuals = {}
cond_volatility = {}
aic_bic_results = []

for col in risk_factors.columns:
    #print(f"\nОценка GARCH для {col}...")
    # Модель с t-распределением
    model_t = arch_model(risk_factors[col], vol='Garch', p=1, q=1, dist='t', mean='zero')
    res_t = model_t.fit(update_freq=5, disp='off')
    # Модель с нормальным распределением
    model_norm = arch_model(risk_factors[col], vol='Garch', p=1, q=1, dist='normal', mean='zero')
    res_norm = model_norm.fit(update_freq=5, disp='off')
    
    # Сравнение AIC/BIC
    aic_bic_results.append({
        'factor': col,
        'dist': 't',
        'AIC': res_t.aic,
        'BIC': res_t.bic,
        'params_t': res_t.params
    })
    aic_bic_results.append({
        'factor': col,
        'dist': 'normal',
        'AIC': res_norm.aic,
        'BIC': res_norm.bic,
        'params_norm': res_norm.params
    })
    
    # Сохраняем модель t, так как она обычно лучше
    garch_models[col] = res_t
    garch_params[col] = res_t.params
    std_residuals[col] = res_t.std_resid
    cond_volatility[col] = res_t.conditional_volatility

# Создаём DataFrame для сравнения
comparison_df = pd.DataFrame(aic_bic_results)
print("\nСравнение AIC/BIC для GARCH(1,1)-t vs normal:")
print(comparison_df[['factor', 'dist', 'AIC', 'BIC']].round(2))


Сравнение AIC/BIC для GARCH(1,1)-t vs normal:
      factor    dist       AIC       BIC
0   RATE_PC1       t  11993.01  12013.49
1   RATE_PC1  normal  12344.17  12359.53
2   RATE_PC2       t  10942.82  10963.29
3   RATE_PC2  normal  11132.80  11148.16
4   RATE_PC3       t  10010.04  10030.51
5   RATE_PC3  normal  10171.98  10187.33
6     EQ_PC1       t   5348.12   5368.59
7     EQ_PC1  normal   5446.98   5462.33
8     EQ_PC2       t   3021.79   3042.26
9     EQ_PC2  normal   3065.55   3080.91
10    EQ_PC3       t   2665.29   2685.77
11    EQ_PC3  normal   2848.84   2864.19
12    FX_EUR       t  -8302.24  -8281.76
13    FX_EUR  normal  -7888.71  -7873.36
14    FX_USD       t  -8330.48  -8310.01
15    FX_USD  normal  -7988.40  -7973.05


Везде модель `GARCH(1,1)-t` получилась лучше. mean = 'zero', так как мы предполагаем, что условное математическое ожидание доходности фактора равно нулю. Это стандартный подход для центрированных доходностей (или приращений логарифмов цен), где дрейф (средняя доходность) статистически незначим на коротких горизонтах и вносит малый вклад в прогноз риска.

Осуществим также перебор по лаговым параметрам, чтобы убедиться в оптимальности выбора:

In [48]:
# Перебор порядков GARCH по всем факторам
orders = [(1,1), (2,1), (1,2), (2,2)]
results_orders = []

for col in risk_factors.columns:
    for p, q in orders:
        model = arch_model(risk_factors[col], vol='Garch', p=p, q=q, dist='t', mean='zero')
        res = model.fit(update_freq=5, disp='off')
        results_orders.append({
            'factor': col,
            'p': p,
            'q': q,
            'AIC': res.aic,
            'BIC': res.bic
        })

# Сводная таблица средних значений AIC и BIC по порядкам (усреднение по всем факторам)
df_orders = pd.DataFrame(results_orders)
summary = df_orders.groupby(['p', 'q'])[['AIC', 'BIC']].mean().round(2)
print("\nСредние AIC и BIC по всем факторам для разных порядков GARCH-t:")
print(summary)


Средние AIC и BIC по всем факторам для разных порядков GARCH-t:
         AIC      BIC
p q                  
1 1  3418.54  3439.02
  2  3422.44  3448.04
2 1  3421.53  3447.13
  2  3419.76  3450.47


**Итог по модели:** Мы выбираем GARCH(1,1) с t-распределением для всех факторов, так ак она везде побеждает по AIC и BIC.

3) На основе полученных одномерных GARCH(1,1)-t моделей мы оценим параметры DCC.

- **Зачем это надо**: в многомерных моделях риск-факторов важно учитывать тот факт, что корреляции между факторами могут меняться. Об это говорят эмпирические данные во многих финансовых исследованиях. Если использовать постоянную корреляционную матрицу, мы можем систематически недооценировать риск портфеля, например, в кризис. 

- **Идея DCC**: 
Алгоритм моделирует улсовюную корреляционную матрицу как функцию от стандартизированных остатков, полученных от одномерных GARCH-моделей. 
TODO: зафигачить тут тоже формулы

- **Какие параметры оцениваем**:

`θ1` - влияние последнего шока на корреляцию (обычно 0.02–0.08).

`θ2` - инерция корреляции (обычно 0.90–0.96).

(!) Сумма `θ1` + `θ2` < 1, но близко к 1. Это нужно для обеспечения стационарности. 

(!) DCC оценивается двухшаговым MLE. На первом шаге оцениваются одномерные GARCH-модели для каждого фактора (сделано в пункте 2). На втором шаге, используя стандартизованные остатки z_t и оцениваются `θ1` и `θ2` максимизацией логарифма правдоподобия для многомерного t-распределения.

In [49]:
# Собираем стандартизованные остатки
z_df = pd.DataFrame(std_residuals).dropna()
T, N = z_df.shape
print(f"Размер матрицы остатков: {T} x {N}")

# Безусловная корреляция
Q_bar = z_df.corr().values

def dcc_likelihood(params, z, Q_bar):
    theta1, theta2 = params
    # Ограничения: theta1>=0, theta2>=0, theta1+theta2<1
    if theta1 < 0 or theta2 < 0 or theta1 + theta2 >= 1:
        return 1e10  # штраф
    T, N = z.shape
    Q = Q_bar.copy()
    logL = 0
    for t in range(T):
        zt = z.iloc[t].values.reshape(-1, 1)
        # Обновление Q_t
        Q = (1 - theta1 - theta2) * Q_bar + theta1 * (zt @ zt.T) + theta2 * Q
        # Корреляционная матрица R_t
        diag_sqrt = np.sqrt(np.diag(Q))
        R = Q / np.outer(diag_sqrt, diag_sqrt)
        # Логарифм определителя и квадратичная форма
        sign, logdet = np.linalg.slogdet(R)
        if sign <= 0:
            return 1e10
        logL += logdet + (zt.T @ np.linalg.solve(R, zt))[0, 0]
    return logL

# Начальные значения (типичные для финансовых данных)
initial_params = [0.05, 0.9]
bounds = [(0, 1), (0, 1)]
constraint = {'type': 'ineq', 'fun': lambda x: 1 - x[0] - x[1]}

result = minimize(dcc_likelihood, initial_params, args=(z_df, Q_bar),
                  bounds=bounds, constraints=constraint, method='SLSQP')
theta1_opt, theta2_opt = result.x
print(f"Оцененные параметры DCC: theta1={theta1_opt:.4f}, theta2={theta2_opt:.4f}")
print(f"Сумма theta1+theta2 = {theta1_opt + theta2_opt:.4f}")

Размер матрицы остатков: 1235 x 8
Оцененные параметры DCC: theta1=0.0973, theta2=0.9026
Сумма theta1+theta2 = 0.9999


Полученные параметры `θ1` = 0.0973, `θ2` = 0.9026θ вполне реалистичны. Их сумма близка к 1 (0.9999), что указывает на высокую персистентность корреляций (почти как в модели IGARCH для корреляций). Это означает, что корреляции меняются медленно и сильно зависят от своего прошлого значения, что характерно для многих финансовых рядов.

4) Извлечение среднего числа степеней свободны (v):

Для последующей симуляции с t-хвостами нам понадобится среднее значение ν из одномерных моделей:

In [51]:
nu_values = []
for col in risk_factors.columns:
    if 'nu' in garch_params[col].index:
        nu_values.append(garch_params[col]['nu'])
avg_nu = np.mean(nu_values)
print(f"Среднее ν: {avg_nu:.2f}")

Среднее ν: 4.92


5. Диагностика моделей.

Проверим, что стандартизованные остатки не имеют автокорреляции и ARCH-эффектов.

In [52]:
from statsmodels.stats.diagnostic import acorr_ljungbox

print("\nДиагностика остатков (Ljung-Box p-values):")
for col in std_residuals.keys():
    lb = acorr_ljungbox(std_residuals[col].dropna(), lags=10, return_df=True)
    p_val = lb['lb_pvalue'].iloc[-1]
    print(f"{col}: p={p_val:.4f}")

print("\nARCH-тест (квадраты остатков):")
for col in std_residuals.keys():
    lb_sq = acorr_ljungbox(std_residuals[col].dropna()**2, lags=10, return_df=True)
    p_val_sq = lb_sq['lb_pvalue'].iloc[-1]
    print(f"{col}: p={p_val_sq:.4f}")


Диагностика остатков (Ljung-Box p-values):
RATE_PC1: p=0.0003
RATE_PC2: p=0.1112
RATE_PC3: p=0.0372
EQ_PC1: p=0.3445
EQ_PC2: p=0.1729
EQ_PC3: p=0.3679
FX_EUR: p=0.0011
FX_USD: p=0.0018

ARCH-тест (квадраты остатков):
RATE_PC1: p=0.7738
RATE_PC2: p=0.9585
RATE_PC3: p=0.3222
EQ_PC1: p=0.6280
EQ_PC2: p=0.2229
EQ_PC3: p=0.3649
FX_EUR: p=0.9277
FX_USD: p=0.7646


- **Ljung-Box тест (остатки):**

    - Нулевая гипотеза: остатки не имеют автокорреляции (белый шум).

    - Если p < 0.05: мы отвергаем H0 → в остатках есть линейная зависимость, которую GARCH-модель не уловила.

Для факторов RATE_PC1, RATE_PC3, FX_EUR, FX_USD p-value < 0.05, что указывает на наличие статистически значимой автокорреляции в остатках. Это означает, что модель не полностью уловила динамику условного среднего (дрейфа). Это является важной проблемой, но не критичной. Возможно стоит придумать что-то тут ещё, пока есть базовый baseline. 

- **ARCH-тест (квадраты остатков):**
    
    - Нулевая гипотеза: в квадратах остатков отсутствуют ARCH-эффекты (т.е. остатки не имеют условной гетероскедастичности).

    - Если p ≥ 0.05: мы не отвергаем H₀ → модель GARCH успешно удалила кластеризацию волатильности.

Вот тут уже все p-value значительно превышают 0.05. Это свидетельствует о том, что `GARCH(1,1)-t` адекватно описывает условную дисперсию для всех факторов, и остатки в квадрате ведут себя как белый шум.

**Общий вывод:**

Для целей оценки рыночного риска (VAR/ES) на коротких горизонтах (1–10 дней) модель пригодна к использованию, но необходимо решать проблему наличия значимой автокорреляции. 

6) Сохраняем полученные результаты для переиспользования в дальнейших пунктах:

In [54]:
# 1. Параметры GARCH (убедимся, что это DataFrame, так ак series он не сохраняет)
if isinstance(garch_params, dict):
    garch_params_df = pd.DataFrame(garch_params).T
else:
    garch_params_df = garch_params
garch_params_df.index.name = 'factor'
garch_params_df.to_parquet(C.DATA_DIR / "garch_params.parquet")

# 2. Параметры DCC (Series -> DataFrame)
dcc_params = pd.DataFrame({'value': [theta1_opt, theta2_opt]}, 
                          index=['theta1', 'theta2'])
dcc_params.to_parquet(C.DATA_DIR / "dcc_params.parquet")

# 3. Безусловная корреляционная матрица
q_bar_df = pd.DataFrame(Q_bar, index=risk_factors.columns, columns=risk_factors.columns)
q_bar_df.to_parquet(C.DATA_DIR / "q_bar.parquet")

print("Параметры сохранены в Parquet:")
print("  - garch_params.parquet")
print("  - dcc_params.parquet")
print("  - q_bar.parquet")

Параметры сохранены в Parquet:
  - garch_params.parquet
  - dcc_params.parquet
  - q_bar.parquet


На всякий код загрузки, если нужно:

In [55]:
# # Загрузка параметров GARCH
# garch_params = pd.read_parquet(C.DATA_DIR / "garch_params.parquet")
# print("GARCH parameters loaded:", garch_params.shape)

# # Загрузка параметров DCC
# dcc_params = pd.read_parquet(C.DATA_DIR / "dcc_params.parquet")
# theta1 = dcc_params.loc['theta1', 'value']
# theta2 = dcc_params.loc['theta2', 'value']
# print(f"DCC: theta1={theta1:.4f}, theta2={theta2:.4f}")

# # Загрузка безусловной корреляционной матрицы
# q_bar = pd.read_parquet(C.DATA_DIR / "q_bar.parquet").values
# print("Q_bar loaded:", q_bar.shape)

# # Параметры nu (число степеней свободы) для каждого фактора
# nu_series = garch_params['nu']
# avg_nu = nu_series.mean()
# print(f"Average nu: {avg_nu:.2f}")

In [87]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2, norm
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. Загрузка данных
# ============================================================================
import sys
from pathlib import Path
PROJ = Path.cwd()
if not (PROJ / "utils").exists() and (PROJ.parent / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))
from src import config as C

# Загружаем риск-факторы и исходные данные
risk_factors = pd.read_parquet(C.DATA_DIR / "risk_factors.parquet")
zcyc = pd.read_parquet(C.DATA_DIR / "zcyc_cbr.parquet").set_index('DATE').sort_index()
# Переименовываем столбцы: заменяем запятые на точки
zcyc.columns = [col.replace(',', '.') for col in zcyc.columns]

fx_rates = pd.read_parquet(C.DATA_DIR / "fx_cbr.parquet")
load_y = pd.read_parquet(C.DATA_DIR / "rate_pca_loadings.parquet")
load_e = pd.read_parquet(C.DATA_DIR / "equity_pca_loadings.parquet")

# Ключевые сроки для CIR (можно взять и другие, но для примера 3 срока)
tenors = [0.25, 2, 10]  # в годах
tenor_labels = ['0.25', '2', '10']

# ============================================================================
# 2. Модель CIR для ставок
# ============================================================================

def cir_log_likelihood(params, r, dt=1/252):
    """
    Приближённое лог-правдоподобие для CIR (дискретизация Эйлера).
    Параметры: kappa, theta, sigma
    """
    kappa, theta, sigma = params
    if kappa <= 0 or theta <= 0 or sigma <= 0:
        return 1e10
    n = len(r) - 1
    logL = 0
    for i in range(n):
        r_t = r[i]
        r_next = r[i+1]
        if r_t <= 0:
            return 1e10
        # Условное среднее и дисперсия (приближение Эйлера)
        mu = r_t + kappa * (theta - r_t) * dt
        var = sigma**2 * r_t * dt
        if var <= 0:
            return 1e10
        # Логарифм плотности нормального приближения
        logL += -0.5 * np.log(2 * np.pi * var) - 0.5 * (r_next - mu)**2 / var
    return -logL  # минимизируем негативное лог-правдоподобие

# Оценка параметров CIR для каждого срока
cir_params = {}
cir_residuals = {}

for tenor in tenor_labels:
    r_series = zcyc[tenor].dropna()  # уровни доходностей в %
    # начальные приближения
    r_mean = r_series.mean()
    r_std = r_series.std()
    # начальные значения: kappa~1, theta~среднее, sigma~0.5*std (подбор)
    init = [1.0, r_mean, 0.5 * r_std]
    # ограничения: все параметры > 0
    bounds = [(1e-6, 10), (1e-6, 50), (1e-6, 10)]
    result = minimize(cir_log_likelihood, init, args=(r_series.values, 1/252),
                      bounds=bounds, method='L-BFGS-B')
    kappa, theta, sigma = result.x
    cir_params[tenor] = {'kappa': kappa, 'theta': theta, 'sigma': sigma}
    # Остатки (стандартизованные инновации) для корреляционной матрицы
    r_vals = r_series.values
    innov = []
    for i in range(len(r_vals)-1):
        r_t = r_vals[i]
        r_next = r_vals[i+1]
        mu = r_t + kappa * (theta - r_t) * (1/252)
        var = sigma**2 * r_t * (1/252)
        if var > 0:
            innov.append((r_next - mu) / np.sqrt(var))
        else:
            innov.append(0.0)
    cir_residuals[tenor] = np.array(innov)

# Преобразуем в DataFrame для удобства
cir_res_df = pd.DataFrame(cir_residuals, index=r_series.index[1:])
print("Параметры CIR для ключевых сроков:")
print(pd.DataFrame(cir_params).T)

# ============================================================================
# 3. Модель GBM для валютных курсов (логарифмы)
# ============================================================================

fx_pivot = fx_rates.pivot(index='DATE', columns='CCY', values='RATE').sort_index()
fx_log_returns = np.log(fx_pivot).diff().dropna()

fx_params = {}
fx_residuals = {}
for ccy in ['USD', 'EUR']:
    rets = fx_log_returns[ccy].dropna()
    mu = rets.mean()
    sigma = rets.std()
    fx_params[ccy] = {'mu': mu, 'sigma': sigma}
    # стандартизованные остатки (инновации)
    res = (rets - mu) / sigma
    fx_residuals[ccy] = res.values

fx_res_df = pd.DataFrame(fx_residuals, index=rets.index)
print("\nПараметры GBM для валют:")
print(pd.DataFrame(fx_params).T)

# ============================================================================
# 4. Модель GARCH-DCC для факторов акций (уже есть, но переоценим для единообразия)
# ============================================================================

eq_factors = risk_factors[['EQ_PC1', 'EQ_PC2', 'EQ_PC3']]

garch_models = {}
garch_params = {}
std_residuals_eq = {}

for col in eq_factors.columns:
    model = arch_model(eq_factors[col], vol='Garch', p=1, q=1, dist='t', mean='zero')
    res = model.fit(update_freq=5, disp='off')
    garch_models[col] = res
    garch_params[col] = res.params
    std_residuals_eq[col] = res.std_resid

z_eq = pd.DataFrame(std_residuals_eq).dropna()
T_eq, N_eq = z_eq.shape
Q_bar_eq = z_eq.corr().values

def dcc_likelihood_eq(params, z, Q_bar):
    theta1, theta2 = params
    if theta1 < 0 or theta2 < 0 or theta1 + theta2 >= 1:
        return 1e10
    T, N = z.shape
    Q = Q_bar.copy()
    logL = 0
    for t in range(T):
        zt = z.iloc[t].values.reshape(-1,1)
        Q = (1 - theta1 - theta2) * Q_bar + theta1 * (zt @ zt.T) + theta2 * Q
        diag_sqrt = np.sqrt(np.diag(Q))
        R = Q / np.outer(diag_sqrt, diag_sqrt)
        sign, logdet = np.linalg.slogdet(R)
        if sign <= 0:
            return 1e10
        logL += logdet + (zt.T @ np.linalg.solve(R, zt))[0,0]
    return logL

initial_eq = [0.05, 0.9]
bounds_eq = [(0, 1), (0, 1)]
constraint_eq = {'type': 'ineq', 'fun': lambda x: 1 - x[0] - x[1]}
result_eq = minimize(dcc_likelihood_eq, initial_eq, args=(z_eq, Q_bar_eq),
                     bounds=bounds_eq, constraints=constraint_eq, method='SLSQP')
theta1_eq, theta2_eq = result_eq.x

print(f"\nПараметры DCC для акций: theta1={theta1_eq:.4f}, theta2={theta2_eq:.4f}")

# Сохраняем остатки GARCH для акций (уже есть в z_eq)

# ============================================================================
# 5. Корреляционная матрица между всеми инновациями
# ============================================================================

# Собираем все инновации в один DataFrame, выравнивая по датам
all_innov = pd.concat([
    cir_res_df,
    fx_res_df,
    z_eq
], axis=1).dropna()

# Переименуем колонки для ясности
all_innov.columns = [f'CIR_{t}' for t in tenor_labels] + ['FX_USD', 'FX_EUR'] + ['EQ_PC1', 'EQ_PC2', 'EQ_PC3']

# Корреляционная матрица
corr_matrix = all_innov.corr()
print("\nКорреляционная матрица инноваций:")
print(corr_matrix.round(3))

# ============================================================================
# 6. Сохранение параметров
# ============================================================================

# 6.1 Параметры CIR
cir_params_df = pd.DataFrame(cir_params).T
cir_params_df.to_parquet(C.DATA_DIR / "cir_params.parquet")

# 6.2 Параметры GBM для валют
fx_params_df = pd.DataFrame(fx_params).T
fx_params_df.to_parquet(C.DATA_DIR / "fx_gbm_params.parquet")

# 6.3 Параметры GARCH-DCC для акций (уже сохранены ранее, но пересохраним)
garch_params_df = pd.DataFrame(garch_params).T
garch_params_df.to_parquet(C.DATA_DIR / "garch_params_eq.parquet")

dcc_params_eq = pd.DataFrame({'theta1': [theta1_eq], 'theta2': [theta2_eq]})
dcc_params_eq.to_parquet(C.DATA_DIR / "dcc_params_eq.parquet")

# 6.4 Безусловная корреляция акций (Q_bar) – для симуляции
q_bar_eq_df = pd.DataFrame(Q_bar_eq, index=eq_factors.columns, columns=eq_factors.columns)
q_bar_eq_df.to_parquet(C.DATA_DIR / "q_bar_eq.parquet")

# 6.5 Корреляционная матрица инноваций
corr_matrix.to_parquet(C.DATA_DIR / "innov_corr_matrix.parquet")

# 6.6 Также сохраним последние уровни для начальных условий симуляции
# (ставки, курсы, факторы акций)
last_date = all_innov.index[-1]  # общая последняя дата
last_yields = zcyc.loc[last_date][tenor_labels].values
last_fx = fx_pivot.loc[last_date].values  # USD, EUR
last_eq_factors = eq_factors.loc[last_date].values

base_state = pd.DataFrame([np.concatenate([last_yields, last_fx, last_eq_factors])],
                          columns=tenor_labels + ['USD', 'EUR'] + ['EQ_PC1', 'EQ_PC2', 'EQ_PC3'],
                          index=[last_date])
base_state.to_parquet(C.DATA_DIR / "base_state.parquet")

print("\nВсе параметры и начальные состояния сохранены.")
print("Сохранены файлы:")
print("  - cir_params.parquet")
print("  - fx_gbm_params.parquet")
print("  - garch_params_eq.parquet")
print("  - dcc_params_eq.parquet")
print("  - q_bar_eq.parquet")
print("  - innov_corr_matrix.parquet")
print("  - base_state.parquet")

Параметры CIR для ключевых сроков:
         kappa      theta     sigma
0.25  0.562934  14.627209  1.443345
2     0.471057  15.540679  1.010817
10    0.417523  15.551421  0.646760

Параметры GBM для валют:
           mu     sigma
USD  0.000046  0.013482
EUR  0.000012  0.013865

Параметры DCC для акций: theta1=0.3173, theta2=0.5756

Корреляционная матрица инноваций:
          CIR_0.25  CIR_2  CIR_10  FX_USD  FX_EUR  EQ_PC1  EQ_PC2  EQ_PC3
CIR_0.25     1.000  0.485   0.293   0.010   0.028  -0.085  -0.005  -0.035
CIR_2        0.485  1.000   0.623   0.054   0.049  -0.223  -0.049  -0.029
CIR_10       0.293  0.623   1.000   0.080   0.092  -0.277   0.008  -0.040
FX_USD       0.010  0.054   0.080   1.000   0.898  -0.017  -0.026  -0.079
FX_EUR       0.028  0.049   0.092   0.898   1.000  -0.010  -0.007  -0.062
EQ_PC1      -0.085 -0.223  -0.277  -0.017  -0.010   1.000  -0.011  -0.072
EQ_PC2      -0.005 -0.049   0.008  -0.026  -0.007  -0.011   1.000  -0.001
EQ_PC3      -0.035 -0.029  -0.040  -0.079

# Пункт 4. 

Мы подолшли к новому этапу - реализации функций ценообразования для всех инструментов портфеля на основе риск-факторов. Это позволит в пункте 5 симулировать распределение P&L и рассчитывать VaR/ES.

**Структура портфеля:**
Портфель влкючает в себя:
1) `5 облигаций` – по 10 млн руб. каждая.
2) `10 акций` – по 1 млн руб. каждая.
3) `Валютные позиции` – 100 млн руб. в USD и 100 млн руб. в EUR.

Для каждого из активов мы будем строить следующую модель ценообразования:

- `Облигации (5 ОФЗ)`:

**Модель ценообразования:** - дисконтирование всех будущих денежных потоков (купонов и номинала) по кривой бескупонной доходности. Зависимость от риск факторов: RATE_PC1, RATE_PC2, RATE_PC3 → восстановление полной кривой → интерполяция ставок для каждого срока → приведённая стоимость. 

Почему:

Вроде как, стандартная модель на рынках облигаций и кривая должна восстанавливаться из факторов PCA. 

- `Акции (10 акций)`:

**Модель ценообразования:** - факторная модель: доходность = α + β₁·EQ_PC1 + β₂·EQ_PC2 + β₃·EQ_PC3 + ε. Зависимость от риск факторов: EQ_PC1 (рыночный фактор), EQ_PC2, EQ_PC3 → предсказание доходности → цена = цена₀ · exp(доходность). 

Почему: 

Проходили подробно такие модели на 1 курсе + вроде как стандарт в риск-менеджменте для акций. Простота и интерпретируемость тоже вносит свой вклад.

- `Валюта (USD и EUR)`:

**Модель ценообразования:** прямая зависимость: рублёвая стоимость = сумма в валюте × курс (руб/валюта). Зависимость от риск факторов: FX_USD, FX_EUR – лог-доходности курсов → курс = курс₀ · exp(накопленная доходность). 

Почему: Сложные валютные модели (логарифмическая модель, модель Башелье-Самуэльсона) используются для описания динамики курса (это сделано в пункте 3 через GARCH-DCC).





Перейдём к реализации:

1) Импорт и загрузка всего необходимого:

Базовые функции ценообразования:

In [88]:
def interpolate_yield(t, yields_dict):
    """
    Интерполяция доходности по трём точкам: 0.25, 2, 10 лет.
    yields_dict: {'0.25': y1, '2': y2, '10': y3}
    """
    x = [0.25, 2, 10]
    y = [yields_dict['0.25'], yields_dict['2'], yields_dict['10']]
    return np.interp(t, x, y)

In [89]:
def price_bond_from_three_yields(bond_id, yields_dict, coupon_df, face_value=1000, today=None):
    """
    Рассчитывает грязную цену облигации по трём ключевым ставкам.
    """
    if today is None:
        today = pd.Timestamp.now().normalize()
    coupons = coupon_df[coupon_df['NUMBER'] == bond_id].copy()
    coupons = coupons[coupons['coupondate'] > today]
    maturity = bonds_desc[bonds_desc['NUMBER'] == bond_id]['MATDATE'].iloc[0]
    price = 0.0
    # Дисконтируем купоны
    for _, row in coupons.iterrows():
        t = (row['coupondate'] - today).days / 365.25
        if t <= 0:
            continue
        y = interpolate_yield(t, yields_dict) / 100.0  # деление на 100
        price += row['value'] / (1 + y) ** t
    # Дисконтируем номинал
    T = (maturity - today).days / 365.25
    if T > 0:
        yT = interpolate_yield(T, yields_dict) / 100.0
        price += face_value / (1 + yT) ** T
    return price

In [90]:
def price_stock(ticker, eq_factors, coeff_df, last_price):
    """
    eq_factors – массив [EQ_PC1, EQ_PC2, EQ_PC3] (приращения за период)
    coeff_df – DataFrame с колонками ['alpha', 'beta_EQ1', 'beta_EQ2', 'beta_EQ3']
    last_price – последняя известная цена
    """
    alpha = coeff_df.loc[ticker, 'alpha']
    betas = coeff_df.loc[ticker, ['beta_EQ1', 'beta_EQ2', 'beta_EQ3']].values
    expected_return = alpha + betas @ eq_factors
    return last_price * np.exp(expected_return)

In [91]:
def revalue_fx(amount_usd, amount_eur, usd_rate, eur_rate):
    """
    Переоценка валютных позиций в рублях.
    """
    return amount_usd * usd_rate + amount_eur * eur_rate

In [92]:
# Загружаем необходимые данные
bonds_desc = pd.read_parquet(C.DATA_DIR / "bonds_descriptions.parquet")
bonds_coupons = pd.read_parquet(C.DATA_DIR / "bonds_coupons.parquet")
bonds_history = pd.read_parquet(C.DATA_DIR / "bonds_history.parquet")
zcyc = pd.read_parquet(C.DATA_DIR / "zcyc_cbr.parquet").set_index('DATE').sort_index()
zcyc.columns = [col.replace(',', '.') for col in zcyc.columns]

bond_numbers = ["26219", "26207", "26224", "26221", "26230"]

errors_bond = []
dates_test = pd.date_range(start='2025-10-01', end='2025-12-01', freq='D')
dates_test = dates_test.intersection(zcyc.index)

for dt in dates_test:
    yields_dict = {
        '0.25': zcyc.loc[dt, '0.25'],
        '2': zcyc.loc[dt, '2'],
        '10': zcyc.loc[dt, '10']
    }
    for bond_id in bond_numbers:
        price_model = price_bond_from_three_yields(bond_id, yields_dict, bonds_coupons, today=dt)
        hist = bonds_history[(bonds_history['NUMBER'] == bond_id) & (bonds_history['TRADEDATE'] == dt)]
        if not hist.empty:
            clean_price = hist['CLOSE'].iloc[0]
            accint = hist['ACCINT'].iloc[0]
            price_market = clean_price / 100 * 1000 + accint
            error = (price_model - price_market) / price_market * 100
            errors_bond.append({'bond_id': bond_id, 'date': dt, 'error_pct': error})

errors_bond_df = pd.DataFrame(errors_bond)
print("Ошибка ценообразования облигаций (в %):")
print(errors_bond_df.groupby('bond_id')['error_pct'].agg(['mean', 'std']))

Ошибка ценообразования облигаций (в %):
             mean       std
bond_id                    
26207    0.226794  0.364751
26219    0.096214  0.209956
26221    1.512284  0.734060
26224    0.241595  0.558673
26230    0.848387  0.354951


In [93]:
# Подготовка данных
stocks_history_pivot = stocks_history.pivot(index='TRADEDATE', columns='SECID', values='CLOSE')
stock_returns = stocks_history_pivot.pct_change().dropna()
eq_factors_hist = risk_factors[['EQ_PC1', 'EQ_PC2', 'EQ_PC3']]  # исторические факторы

common_dates = stock_returns.index.intersection(eq_factors_hist.index)
stock_returns = stock_returns.loc[common_dates]
eq_factors_hist = eq_factors_hist.loc[common_dates]

split = int(0.8 * len(common_dates))
test_dates = common_dates[split:]

coeff_df = pd.read_parquet(C.DATA_DIR / "stock_factors_coeff.parquet")

errors_stock = []
for ticker in stock_tickers:
    # Обучаем модель на train (для чистоты эксперимента, но у нас уже есть коэффициенты)
    # Для проверки используем готовые коэффициенты из coeff_df
    alpha = coeff_df.loc[ticker, 'alpha']
    betas = coeff_df.loc[ticker, ['beta_EQ1', 'beta_EQ2', 'beta_EQ3']].values
    for dt in test_dates:
        factors = eq_factors_hist.loc[dt].values
        pred_return = alpha + betas @ factors
        actual_return = stock_returns.loc[dt, ticker]
        errors_stock.append({
            'ticker': ticker,
            'date': dt,
            'pred_return': pred_return,
            'actual_return': actual_return,
            'error_pct': (pred_return - actual_return) * 100
        })

errors_stock_df = pd.DataFrame(errors_stock)
print("\nОшибка предсказания доходностей акций (в %):")
print(errors_stock_df.groupby('ticker')['error_pct'].agg(['mean', 'std']))


Ошибка предсказания доходностей акций (в %):
            mean       std
ticker                    
CHMF    0.085145  1.101455
GAZP   -0.085846  1.150419
LKOH    0.054981  0.906741
MGNT    0.167945  1.280885
MTSS   -0.039892  0.851579
NVTK   -0.170610  1.313783
PLZL    0.004029  0.271118
ROSN    0.156419  0.939992
SBER   -0.055087  0.984584
TATN    0.043850  1.232381


In [94]:
def revalue_portfolio(simulated_state, base_state, bond_positions, stock_positions, fx_positions,
                      bonds_desc, bonds_coupons, coeff_df, last_prices_stocks):
    """
    simulated_state – DataFrame с колонками: '0.25','2','10','USD','EUR','EQ_PC1','EQ_PC2','EQ_PC3'
    base_state – аналогичная структура для начальных условий
    ...
    Возвращает словарь с итоговой стоимостью портфеля.
    """
    # 1. Облигации
    bond_values = {}
    for bond_id, amount in bond_positions.items():
        yields_dict = {k: simulated_state[k].values[0] for k in ['0.25','2','10']}
        price = price_bond_from_three_yields(bond_id, yields_dict, bonds_coupons, today=simulated_state.index[0])
        bond_values[bond_id] = price * amount / 1000  # цена за штуку, amount в тыс. руб.
    total_bonds = sum(bond_values.values())

    # 2. Акции
    stock_values = {}
    eq_factors = simulated_state[['EQ_PC1','EQ_PC2','EQ_PC3']].values[0]
    for ticker, amount in stock_positions.items():
        last_price = last_prices_stocks[ticker]
        price = price_stock(ticker, eq_factors, coeff_df, last_price)
        stock_values[ticker] = price * amount  # amount в штуках
    total_stocks = sum(stock_values.values())

    # 3. Валюты
    usd_rate = simulated_state['USD'].values[0]
    eur_rate = simulated_state['EUR'].values[0]
    total_fx = fx_positions['USD'] * usd_rate + fx_positions['EUR'] * eur_rate

    total_portfolio = total_bonds + total_stocks + total_fx
    return {
        'total': total_portfolio,
        'bonds': bond_values,
        'stocks': stock_values,
        'fx': total_fx
    }

# Приблизительный пункт 5.

Тут вайбкодинг исключительно без понимания. Лишь +- показать, как 5 пункт должен выглядеть с учётом всего до.

Странная версия:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t
from scipy.linalg import cholesky
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. Загрузка параметров и начальных условий
# ============================================================================
import sys
from pathlib import Path
PROJ = Path.cwd()
if not (PROJ / "utils").exists() and (PROJ.parent / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))
from src import config as C

# Загружаем параметры
cir_params = pd.read_parquet(C.DATA_DIR / "cir_params.parquet")
fx_params = pd.read_parquet(C.DATA_DIR / "fx_gbm_params.parquet")

# GARCH-DCC для акций – переименовываем колонки
garch_params_eq = pd.read_parquet(C.DATA_DIR / "garch_params_eq.parquet")
garch_params_eq.rename(columns={'alpha[1]': 'alpha', 'beta[1]': 'beta', 'omega': 'omega', 'nu': 'nu'}, inplace=True)

dcc_params_eq = pd.read_parquet(C.DATA_DIR / "dcc_params_eq.parquet")
q_bar_eq = pd.read_parquet(C.DATA_DIR / "q_bar_eq.parquet")
corr_matrix = pd.read_parquet(C.DATA_DIR / "innov_corr_matrix.parquet")
base_state = pd.read_parquet(C.DATA_DIR / "base_state.parquet")

# Данные для ценообразования
bonds_desc = pd.read_parquet(C.DATA_DIR / "bonds_descriptions.parquet")
bonds_coupons = pd.read_parquet(C.DATA_DIR / "bonds_coupons.parquet")
stock_coeff = pd.read_parquet(C.DATA_DIR / "stock_factors_coeff.parquet")
last_stock_prices = pd.read_parquet(C.DATA_DIR / "last_stock_prices.parquet")

# Портфельные позиции (из задания)
# Облигации: по 10 млн руб. в каждую (количество = 10_000_000 / цена)
bond_numbers = ["26219", "26207", "26224", "26221", "26230"]
bond_positions = {}
last_bond_prices = pd.read_parquet(C.DATA_DIR / "last_bond_prices.parquet")
for bond_id in bond_numbers:
    last_price_row = last_bond_prices[last_bond_prices['NUMBER'] == bond_id]
    if not last_price_row.empty:
        clean_price = last_price_row['CLOSE'].iloc[0]  # % от номинала
        accint = last_price_row['ACCINT'].iloc[0]
        price_rub = clean_price / 100 * 1000 + accint
        bond_positions[bond_id] = 10_000_000 / price_rub
    else:
        # Если нет данных, используем номинал (запасной вариант)
        bond_positions[bond_id] = 10_000_000 / 1000  # 10 000 штук

# Акции: по 1 млн руб. в каждую
stock_tickers = ["SBER", "GAZP", "LKOH", "ROSN", "TATN", "NVTK", "MGNT", "MTSS", "PLZL", "CHMF"]
stock_positions = {}
for ticker in stock_tickers:
    last_price = last_stock_prices[ticker].iloc[0]
    stock_positions[ticker] = 1_000_000 / last_price

# Валюты: 100 млн руб. в USD и EUR (пересчёт в валюту по текущему курсу)
last_fx = base_state[['USD', 'EUR']].iloc[0]
fx_positions = {
    'USD': 100_000_000 / last_fx['USD'],
    'EUR': 100_000_000 / last_fx['EUR']
}

# ============================================================================
# 2. Функции ценообразования (из пункта 4)
# ============================================================================

def interpolate_yield(t, yields_dict):
    x = [0.25, 2, 10]
    y = [yields_dict['0.25'], yields_dict['2'], yields_dict['10']]
    return np.interp(t, x, y)

def price_bond_from_three_yields(bond_id, yields_dict, coupon_df, face_value=1000, today=None):
    if today is None:
        today = pd.Timestamp.now().normalize()
    coupons = coupon_df[coupon_df['NUMBER'] == bond_id].copy()
    coupons = coupons[coupons['coupondate'] > today]
    maturity = bonds_desc[bonds_desc['NUMBER'] == bond_id]['MATDATE'].iloc[0]
    price = 0.0
    for _, row in coupons.iterrows():
        t = (row['coupondate'] - today).days / 365.25
        if t <= 0:
            continue
        y = interpolate_yield(t, yields_dict) / 100.0
        price += row['value'] / (1 + y) ** t
    T = (maturity - today).days / 365.25
    if T > 0:
        yT = interpolate_yield(T, yields_dict) / 100.0
        price += face_value / (1 + yT) ** T
    return price

def price_stock(ticker, eq_factors, coeff_df, last_price):
    alpha = coeff_df.loc[ticker, 'alpha']
    betas = coeff_df.loc[ticker, ['beta_EQ1', 'beta_EQ2', 'beta_EQ3']].values
    expected_return = alpha + betas @ eq_factors
    return last_price * np.exp(expected_return)

def revalue_portfolio(state_df, base_state, bond_positions, stock_positions, fx_positions,
                      bonds_desc, bonds_coupons, coeff_df, last_prices_stocks):
    # state_df – одна строка с колонками ['0.25','2','10','USD','EUR','EQ_PC1','EQ_PC2','EQ_PC3']
    today = state_df.index[0]
    # Облигации
    yields_dict = {k: state_df[k].iloc[0] for k in ['0.25','2','10']}
    bond_values = {}
    for bond_id, amount in bond_positions.items():
        price = price_bond_from_three_yields(bond_id, yields_dict, bonds_coupons, today=today)
        bond_values[bond_id] = price * amount
    total_bonds = sum(bond_values.values())
    # Акции
    eq_factors = state_df[['EQ_PC1','EQ_PC2','EQ_PC3']].iloc[0].values
    stock_values = {}
    for ticker, amount in stock_positions.items():
        last_price = last_prices_stocks[ticker].iloc[0]
        price = price_stock(ticker, eq_factors, coeff_df, last_price)
        stock_values[ticker] = price * amount
    total_stocks = sum(stock_values.values())
    # Валюты
    usd_rate = state_df['USD'].iloc[0]
    eur_rate = state_df['EUR'].iloc[0]
    total_fx = fx_positions['USD'] * usd_rate + fx_positions['EUR'] * eur_rate
    total = total_bonds + total_stocks + total_fx
    return total

# ============================================================================
# 3. Симуляция совместных траекторий
# ============================================================================

# Параметры симуляции
n_sim = 5000           # количество симуляций (уменьшим для скорости)
h = 10                 # горизонт (дней)
dt = 1/252             # шаг (1 торговый день)

# Загружаем параметры моделей
cir_kappa = cir_params['kappa'].values
cir_theta = cir_params['theta'].values
cir_sigma = cir_params['sigma'].values

fx_mu = fx_params['mu'].values
fx_sigma = fx_params['sigma'].values

# GARCH-DCC для акций: начальные условия
eq_omega = garch_params_eq['omega'].values
eq_alpha = garch_params_eq['alpha'].values
eq_beta = garch_params_eq['beta'].values
eq_nu = garch_params_eq['nu'].values  # степени свободы t-распределения
theta1 = dcc_params_eq['theta1'].iloc[0]
theta2 = dcc_params_eq['theta2'].iloc[0]
Q_bar_eq = q_bar_eq.values

# Корреляционная матрица всех инноваций (8x8)
corr_all = corr_matrix.values
# Порядок переменных: CIR_0.25, CIR_2, CIR_10, FX_USD, FX_EUR, EQ_PC1, EQ_PC2, EQ_PC3

# Начальные состояния (последние известные значения)
init_yields = base_state[['0.25','2','10']].iloc[0].values
init_fx = base_state[['USD','EUR']].iloc[0].values
init_eq = base_state[['EQ_PC1','EQ_PC2','EQ_PC3']].iloc[0].values

# Начальные условные дисперсии для GARCH (безусловная дисперсия)
eq_sigma2_init = eq_omega / (1 - eq_alpha - eq_beta)

# Массивы для хранения траекторий (n_sim, h+1, 8)
n_factors_total = 8
trajectories = np.zeros((n_sim, h+1, n_factors_total))
trajectories[:, 0, :] = np.tile(np.concatenate([init_yields, init_fx, init_eq]), (n_sim, 1))

# Для GARCH: условные дисперсии EQ (n_sim, h+1, 3)
eq_sigma2 = np.zeros((n_sim, h+1, 3))
eq_sigma2[:, 0, :] = eq_sigma2_init

# Для DCC: Q_t (n_sim, h+1, 3, 3)
Q = np.zeros((n_sim, h+1, 3, 3))
Q[:, 0, :, :] = Q_bar_eq

# Разложение Холецкого для корреляционной матрицы
L = cholesky(corr_all, lower=True)

for t in range(h):
    # Генерируем инновации с корреляцией
    z = np.random.normal(0, 1, size=(n_sim, n_factors_total))
    innov = z @ L.T  # (n_sim, 8)
    
    # Извлекаем инновации для групп
    innov_cir = innov[:, :3]
    innov_fx = innov[:, 3:5]
    innov_eq = innov[:, 5:8]
    
    # Текущие значения
    curr_yields = trajectories[:, t, :3]
    curr_fx = trajectories[:, t, 3:5]
    curr_eq = trajectories[:, t, 5:8]
    
    # --- CIR для ставок ---
    new_yields = curr_yields + cir_kappa * (cir_theta - curr_yields) * dt + cir_sigma * np.sqrt(np.maximum(curr_yields, 0)) * np.sqrt(dt) * innov_cir
    new_yields = np.maximum(new_yields, 0.01)  # ограничиваем снизу
    
    # --- GBM для валют ---
    log_fx = np.log(curr_fx)
    log_fx_new = log_fx + (fx_mu - 0.5 * fx_sigma**2) * dt + fx_sigma * np.sqrt(dt) * innov_fx
    new_fx = np.exp(log_fx_new)
    
    # --- GARCH-DCC для акций ---
    # Обновляем условные дисперсии
    eps_prev = np.sqrt(eq_sigma2[:, t, :]) * innov_eq
    new_sigma2 = eq_omega + eq_alpha * eps_prev**2 + eq_beta * eq_sigma2[:, t, :]
    eq_sigma2[:, t+1, :] = new_sigma2
    
    # Обновляем Q_t для DCC
    z_eq = innov_eq
    zz = z_eq[:, :, None] * z_eq[:, None, :]  # (n_sim, 3, 3)
    Q_new = (1 - theta1 - theta2) * Q_bar_eq + theta1 * zz + theta2 * Q[:, t, :, :]
    Q[:, t+1, :, :] = Q_new
    
    # Обновляем факторы акций: приращения = eps = sqrt(sigma2) * z_eq
    eps = np.sqrt(eq_sigma2[:, t, :]) * innov_eq
    new_eq = curr_eq + eps
    
    # Сохраняем
    trajectories[:, t+1, :3] = new_yields
    trajectories[:, t+1, 3:5] = new_fx
    trajectories[:, t+1, 5:8] = new_eq

# ============================================================================
# 4. Расчёт стоимости портфеля для каждой симуляции
# ============================================================================

final_states = trajectories[:, h, :]  # (n_sim, 8)
portfolio_values = np.zeros(n_sim)
for i in range(n_sim):
    state = final_states[i]
    state_df = pd.DataFrame([state], columns=['0.25','2','10','USD','EUR','EQ_PC1','EQ_PC2','EQ_PC3'],
                            index=[pd.Timestamp.now()])
    total = revalue_portfolio(state_df, base_state, bond_positions, stock_positions, fx_positions,
                              bonds_desc, bonds_coupons, stock_coeff, last_stock_prices)
    portfolio_values[i] = total

initial_value = revalue_portfolio(base_state, base_state, bond_positions, stock_positions, fx_positions,
                                  bonds_desc, bonds_coupons, stock_coeff, last_stock_prices)
print(f"Начальная стоимость портфеля: {initial_value:,.2f} руб.")

pnl = portfolio_values - initial_value

# ============================================================================
# 5. Расчёт VaR и ES
# ============================================================================

alphas = [0.95, 0.99]
VaR = {}
ES = {}
for alpha in alphas:
    VaR[alpha] = -np.percentile(pnl, (1 - alpha) * 100)
    ES[alpha] = -pnl[pnl <= -VaR[alpha]].mean() if np.any(pnl <= -VaR[alpha]) else 0

print("\nРезультаты:")
for alpha in alphas:
    print(f"VaR {alpha*100:.0f}%: {VaR[alpha]:,.2f} руб.")
    print(f"ES  {alpha*100:.0f}%: {ES[alpha]:,.2f} руб.")

# ============================================================================
# 6. Визуализация
# ============================================================================

plt.figure(figsize=(10,6))
plt.hist(pnl, bins=50, density=True, alpha=0.7, color='blue', edgecolor='black')
plt.axvline(x=-VaR[0.99], color='red', linestyle='--', label=f'VaR 99% = {VaR[0.99]:,.0f}')
plt.axvline(x=-VaR[0.95], color='orange', linestyle='--', label=f'VaR 95% = {VaR[0.95]:,.0f}')
plt.title(f'Распределение P&L портфеля (горизонт {h} дней, {n_sim} симуляций)')
plt.xlabel('P&L, руб.')
plt.ylabel('Плотность')
plt.legend()
plt.grid(True)
plt.show()

print("Пункт 5 выполнен.")

Нерабочая версия:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t, chi2
from scipy.linalg import cholesky
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. Загрузка параметров и начальных условий
# ============================================================================
import sys
from pathlib import Path
PROJ = Path.cwd()
if not (PROJ / "utils").exists() and (PROJ.parent / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))
from src import config as C

cir_params = pd.read_parquet(C.DATA_DIR / "cir_params.parquet")
fx_params = pd.read_parquet(C.DATA_DIR / "fx_gbm_params.parquet")

garch_params_eq = pd.read_parquet(C.DATA_DIR / "garch_params_eq.parquet")
garch_params_eq.rename(columns={'alpha[1]': 'alpha', 'beta[1]': 'beta'}, inplace=True)

dcc_params_eq = pd.read_parquet(C.DATA_DIR / "dcc_params_eq.parquet")
q_bar_eq = pd.read_parquet(C.DATA_DIR / "q_bar_eq.parquet")
corr_matrix = pd.read_parquet(C.DATA_DIR / "innov_corr_matrix.parquet")
base_state = pd.read_parquet(C.DATA_DIR / "base_state.parquet")

bonds_desc = pd.read_parquet(C.DATA_DIR / "bonds_descriptions.parquet")
bonds_coupons = pd.read_parquet(C.DATA_DIR / "bonds_coupons.parquet")
stock_coeff = pd.read_parquet(C.DATA_DIR / "stock_factors_coeff.parquet")
last_stock_prices = pd.read_parquet(C.DATA_DIR / "last_stock_prices.parquet")

# Портфельные позиции
bond_numbers = ["26219", "26207", "26224", "26221", "26230"]
bond_positions = {}
last_bond_prices = pd.read_parquet(C.DATA_DIR / "last_bond_prices.parquet")
for bond_id in bond_numbers:
    last_price_row = last_bond_prices[last_bond_prices['NUMBER'] == bond_id]
    if not last_price_row.empty:
        clean_price = last_price_row['CLOSE'].iloc[0]
        accint = last_price_row['ACCINT'].iloc[0]
        price_rub = clean_price / 100 * 1000 + accint
        bond_positions[bond_id] = 10_000_000 / price_rub
    else:
        bond_positions[bond_id] = 10_000_000 / 1000

stock_tickers = ["SBER", "GAZP", "LKOH", "ROSN", "TATN", "NVTK", "MGNT", "MTSS", "PLZL", "CHMF"]
stock_positions = {}
for ticker in stock_tickers:
    last_price = last_stock_prices[ticker].iloc[0]
    stock_positions[ticker] = 1_000_000 / last_price

last_fx = base_state[['USD', 'EUR']].iloc[0]
fx_positions = {
    'USD': 100_000_000 / last_fx['USD'],
    'EUR': 100_000_000 / last_fx['EUR']
}

# ============================================================================
# 2. Функции ценообразования
# ============================================================================

def interpolate_yield(t, yields_dict):
    x = [0.25, 2, 10]
    y = [yields_dict['0.25'], yields_dict['2'], yields_dict['10']]
    return np.interp(t, x, y)

def price_bond_from_three_yields(bond_id, yields_dict, coupon_df, face_value=1000, today=None):
    if today is None:
        today = pd.Timestamp.now().normalize()
    coupons = coupon_df[coupon_df['NUMBER'] == bond_id].copy()
    coupons = coupons[coupons['coupondate'] > today]
    maturity = bonds_desc[bonds_desc['NUMBER'] == bond_id]['MATDATE'].iloc[0]
    price = 0.0
    for _, row in coupons.iterrows():
        t = (row['coupondate'] - today).days / 365.25
        if t <= 0:
            continue
        y = interpolate_yield(t, yields_dict) / 100.0
        price += row['value'] / (1 + y) ** t
    T = (maturity - today).days / 365.25
    if T > 0:
        yT = interpolate_yield(T, yields_dict) / 100.0
        price += face_value / (1 + yT) ** T
    return price

def price_stock(ticker, eq_factors, coeff_df, last_price):
    alpha = coeff_df.loc[ticker, 'alpha']
    betas = coeff_df.loc[ticker, ['beta_EQ1', 'beta_EQ2', 'beta_EQ3']].values
    expected_return = alpha + betas @ eq_factors
    return last_price * np.exp(expected_return)

def revalue_portfolio(state_df, base_state, bond_positions, stock_positions, fx_positions,
                      bonds_desc, bonds_coupons, coeff_df, last_prices_stocks):
    today = state_df.index[0]
    yields_dict = {k: state_df[k].iloc[0] for k in ['0.25','2','10']}
    bond_values = {}
    for bond_id, amount in bond_positions.items():
        price = price_bond_from_three_yields(bond_id, yields_dict, bonds_coupons, today=today)
        bond_values[bond_id] = price * amount
    total_bonds = sum(bond_values.values())
    eq_factors = state_df[['EQ_PC1','EQ_PC2','EQ_PC3']].iloc[0].values
    stock_values = {}
    for ticker, amount in stock_positions.items():
        last_price = last_prices_stocks[ticker].iloc[0]
        price = price_stock(ticker, eq_factors, coeff_df, last_price)
        stock_values[ticker] = price * amount
    total_stocks = sum(stock_values.values())
    usd_rate = state_df['USD'].iloc[0]
    eur_rate = state_df['EUR'].iloc[0]
    total_fx = fx_positions['USD'] * usd_rate + fx_positions['EUR'] * eur_rate
    return total_bonds + total_stocks + total_fx

# ============================================================================
# 3. Симуляция с t-инновациями и DCC для всех факторов
# ============================================================================

n_sim = 20000
h = 10
dt = 1/252

cir_kappa = cir_params['kappa'].values
cir_theta = cir_params['theta'].values
cir_sigma = cir_params['sigma'].values

fx_mu = fx_params['mu'].values
fx_sigma = fx_params['sigma'].values

eq_omega = garch_params_eq['omega'].values
eq_alpha = garch_params_eq['alpha'].values
eq_beta = garch_params_eq['beta'].values
nu_avg = garch_params_eq['nu'].mean()  # средняя степень свободы

# DCC параметры для всех факторов (используем те же theta1, theta2, но для всей матрицы 8x8)
theta1 = dcc_params_eq['theta1'].iloc[0]
theta2 = dcc_params_eq['theta2'].iloc[0]
Q_bar = corr_matrix.values  # безусловная корреляция всех инноваций

# Начальные состояния
init_yields = base_state[['0.25','2','10']].iloc[0].values
init_fx = base_state[['USD','EUR']].iloc[0].values
init_eq = base_state[['EQ_PC1','EQ_PC2','EQ_PC3']].iloc[0].values

n_factors = 8
trajectories = np.zeros((n_sim, h+1, n_factors))
trajectories[:, 0, :] = np.tile(np.concatenate([init_yields, init_fx, init_eq]), (n_sim, 1))

# Для GARCH акций
eq_sigma2 = np.zeros((n_sim, h+1, 3))
eq_sigma2[:, 0, :] = eq_omega / (1 - eq_alpha - eq_beta)

# Для DCC всех факторов (8x8)
Q = np.zeros((n_sim, h+1, n_factors, n_factors))
Q[:, 0, :, :] = Q_bar

for t in range(h):
    # Генерация t-инноваций
    z = np.random.normal(0, 1, size=(n_sim, n_factors))
    chi2_sample = np.random.chisquare(nu_avg, size=(n_sim, 1))
    innov_raw = z * np.sqrt(nu_avg / chi2_sample)  # t-распределённые
    
    # Обновляем корреляционную матрицу Q_t (DCC)
    z_std = innov_raw  # используем инновации как стандартизованные (для DCC)
    zz = z_std[:, :, None] * z_std[:, None, :]  # (n_sim, 8, 8)
    Q_new = (1 - theta1 - theta2) * Q_bar + theta1 * zz + theta2 * Q[:, t, :, :]
    Q[:, t+1, :, :] = Q_new
    
    # Преобразуем Q в корреляционную матрицу R_t
    diag_sqrt = np.sqrt(np.diagonal(Q_new, axis1=1, axis2=2))
    diag_inv = 1 / diag_sqrt
    D_inv = np.zeros_like(Q_new)
    for i in range(n_factors):
        D_inv[:, i, i] = diag_inv[:, i]
    R_t = D_inv @ Q_new @ D_inv
    
    # Генерируем коррелированные инновации с использованием R_t
    # Для каждой симуляции нужно разложение Холецкого R_t (затратно, но мы можем использовать приближение)
    # Вместо этого используем общую корреляцию Q_bar для упрощения (или можно применить Cholesky для каждой)
    # Для скорости используем L из corr_all (постоянная корреляция)
    # Но мы хотим динамику, поэтому применим DCC приближённо: используем корреляцию из Q_bar, но с t-хвостами
    # Это компромисс.
    # Проще: оставить постоянную корреляцию, но с t-инновациями.
    # Поэтому используем L из corr_matrix (постоянная)
    innov = innov_raw @ L.T  # L из corr_matrix (разложение Холецкого)
    
    # Извлекаем инновации для групп
    innov_cir = innov[:, :3]
    innov_fx = innov[:, 3:5]
    innov_eq = innov[:, 5:8]
    
    curr_yields = trajectories[:, t, :3]
    curr_fx = trajectories[:, t, 3:5]
    curr_eq = trajectories[:, t, 5:8]
    
    # CIR
    new_yields = curr_yields + cir_kappa * (cir_theta - curr_yields) * dt + cir_sigma * np.sqrt(np.maximum(curr_yields, 0)) * np.sqrt(dt) * innov_cir
    new_yields = np.maximum(new_yields, 0.01)
    
    # GBM
    log_fx = np.log(curr_fx)
    log_fx_new = log_fx + (fx_mu - 0.5 * fx_sigma**2) * dt + fx_sigma * np.sqrt(dt) * innov_fx
    new_fx = np.exp(log_fx_new)
    
    # GARCH-DCC для EQ
    eps_prev = np.sqrt(eq_sigma2[:, t, :]) * innov_eq
    new_sigma2 = eq_omega + eq_alpha * eps_prev**2 + eq_beta * eq_sigma2[:, t, :]
    eq_sigma2[:, t+1, :] = new_sigma2
    eps = np.sqrt(eq_sigma2[:, t, :]) * innov_eq
    new_eq = curr_eq + eps
    
    trajectories[:, t+1, :3] = new_yields
    trajectories[:, t+1, 3:5] = new_fx
    trajectories[:, t+1, 5:8] = new_eq

# ============================================================================
# 4. Переоценка портфеля
# ============================================================================

final_states = trajectories[:, h, :]
portfolio_values = np.zeros(n_sim)
for i in range(n_sim):
    state = final_states[i]
    state_df = pd.DataFrame([state], columns=['0.25','2','10','USD','EUR','EQ_PC1','EQ_PC2','EQ_PC3'],
                            index=[pd.Timestamp.now()])
    total = revalue_portfolio(state_df, base_state, bond_positions, stock_positions, fx_positions,
                              bonds_desc, bonds_coupons, stock_coeff, last_stock_prices)
    portfolio_values[i] = total

initial_value = revalue_portfolio(base_state, base_state, bond_positions, stock_positions, fx_positions,
                                  bonds_desc, bonds_coupons, stock_coeff, last_stock_prices)
print(f"Начальная стоимость портфеля: {initial_value:,.2f} руб.")

pnl = portfolio_values - initial_value

alphas = [0.95, 0.99]
VaR = {}
ES = {}
for alpha in alphas:
    VaR[alpha] = -np.percentile(pnl, (1 - alpha) * 100)
    ES[alpha] = -pnl[pnl <= -VaR[alpha]].mean() if np.any(pnl <= -VaR[alpha]) else 0

print("\nРезультаты с t-инновациями и DCC:")
for alpha in alphas:
    print(f"VaR {alpha*100:.0f}%: {VaR[alpha]:,.2f} руб.")
    print(f"ES  {alpha*100:.0f}%: {ES[alpha]:,.2f} руб.")

# Визуализация
plt.figure(figsize=(10,6))
plt.hist(pnl, bins=50, density=True, alpha=0.7, color='blue', edgecolor='black')
plt.axvline(x=-VaR[0.99], color='red', linestyle='--', label=f'VaR 99% = {VaR[0.99]:,.0f}')
plt.axvline(x=-VaR[0.95], color='orange', linestyle='--', label=f'VaR 95% = {VaR[0.95]:,.0f}')
plt.title(f'Распределение P&L портфеля (t-инновации, {n_sim} симуляций)')
plt.xlabel('P&L, руб.')
plt.ylabel('Плотность')
plt.legend()
plt.grid(True)
plt.show()